# Res por canal de venta — corrida semanal

Genera los 8 PowerPoint de Res (2 regiones × 4 canales de venta) para los materiales de `FOCO`: el mes anterior
cerrado contra el mes en curso hasta `CORTE_D`. **Cada semana solo cambie `CORTE_D` en A.1** y ejecute todo en orden
(*Run All*). Solo consulta el mes anterior y el mes en curso; no usa el histórico 2025–2026.

- **A. Preparación**: parámetros, conexión, consulta del Parquet del mes y su lectura.
- **B. Funciones** que usan las láminas.
- **C. Salidas**: los PowerPoint, en `output/Res/`.

# A. Preparación
## A.1 Parámetros

> **`CORTE_D`: último día incluido del mes en curso** (p. ej. `date(2026, 10, 17)`); `None` toma hasta ayer. Con
> `GENERAR_PARQUET_D = True` vuelve a consultar la base aunque el Parquet de ese corte ya exista (para traer lo que
> se haya cargado después).

In [ ]:
from datetime import date, timedelta
from pathlib import Path

AQUI = Path.cwd()
SALIDA = AQUI / "output" / "Res"   # carpeta donde se guardan los resultados de este notebook
SALIDA.mkdir(parents=True, exist_ok=True)

TOP_CLIENTES = 30                     # clientes en los rankings (top de clientes)
TOP_DISTRIBUIDORAS = 30               # distribuidoras en el top del Foco Res

# >>> CORTE DEL MES EN CURSO <<<   ultimo dia incluido; None = ayer (p. ej. date(2026, 10, 11) para otro corte)
CORTE_D = date(2026, 10, 17)
GENERAR_PARQUET_D = True             # True: vuelve a consultar el Parquet del mes en curso aunque ya exista
FOCO = ["Molida-Hamburguesa Res", "Res Ref y Cong"]   # grupos de material 6 del Foco Res

## A.2 Librerías y conexión

Usa la conexión `DB_*` del `.env` de esta carpeta o, si no hay, el del proyecto de lista de precios.

In [ ]:
import base64
import io
import os
from contextlib import contextmanager

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from dotenv import load_dotenv
from IPython.display import HTML, Markdown, display
from sqlalchemy import URL, create_engine, text

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)


def engine():
    load_dotenv(AQUI / ".env") or load_dotenv(AQUI.parent / "Lista de precios" / ".env")
    faltan = [v for v in ("DB_HOST", "DB_NAME", "DB_USER", "DB_PASSWORD") if not os.getenv(v)]
    if faltan:
        raise RuntimeError(f"Faltan en el .env: {', '.join(faltan)}. "
                           f"Copie .env.example como .env en {AQUI} y completelo.")
    url = URL.create("postgresql+psycopg2", username=os.environ["DB_USER"], password=os.environ["DB_PASSWORD"],
                     host=os.environ["DB_HOST"], port=int(os.getenv("DB_PORT", "5432")),
                     database=os.environ["DB_NAME"])
    return create_engine(url, pool_pre_ping=True)


ENGINE = engine()
with ENGINE.connect() as c:
    c.execute(text("SELECT 1"))
print(f"Conexion OK: {ENGINE.url.host}/{ENGINE.url.database}")

## A.3 SQL y funciones del Parquet

El SQL de la tabla base (vista `daily_sales_sector_10_15_20_21` de `commercial_bi`) y las funciones que calculan los %,
los rangos y escriben el Parquet por partes de `FILAS_POR_PARTE` filas; las usa A.4.

- `value` es la venta **neta**; bruta = `value + discount_value`. `discount_value` es el ajuste total.
- Fecha corta = `short_dated_discount`; ajuste de precio de mercado = `discount_value − short_dated_discount`.
- Grano: fecha × cliente × grupo de material 6 × distribuidora × canal × sector. El % de lista y el rango (22
  tramos) se calculan sobre ese grano.

In [ ]:
FILAS_POR_PARTE = 500_000

SQL_DESCUENTOS = """
WITH cliente_attr AS (
    -- Un unico registro por cliente (misma regla que lista de precios:
    -- price_group valido y el menor alfabeticamente).
    SELECT DISTINCT ON (cc.client)
        cc.client,
        NULLIF(cc.price_group, '')       AS price_group,
        cc.name                          AS nombre_cliente,
        NULLIF(cc.customer_typology, '') AS tipologia
    FROM public.commercial_clients cc
    ORDER BY cc.client,
             (cc.price_group IS NULL OR cc.price_group = '') ASC,
             cc.price_group ASC
)
SELECT
    v.date                                           AS fecha,
    v.client                                         AS cliente,
    cp.nombre_cliente,
    cp.price_group                                   AS lista_precio,
    m.gr6                                            AS grupo_material_6,
    g6.name                                          AS nombre_gr6,
    dc.distribution_channel_name                     AS canal_distribucion,
    cdd.new_zone_code                                AS distribuidora,
    regexp_replace(cdd.new_zone_name, '^[^-]*-', '') AS nombre_distribuidora,
    cdd.company_code::text                           AS sociedad,
    -- La tipologia de la venta es la del momento; si viene vacia, la del maestro
    LPAD(COALESCE(NULLIF(v.client_typology, ''), cp.tipologia, '0'), 3, '0') AS tipologia,
    t.name                                           AS nombre_tipologia,
    t.sales_channel_name                             AS canal_venta,
    m.sector,
    SUM(v.kilogram)                                  AS venta_kg,
    SUM(v.value)                                     AS venta_neta,
    SUM(v.value + COALESCE(v.discount_value, 0))     AS venta_bruta,
    SUM(COALESCE(v.discount_value, 0))               AS descuento_total,
    SUM(v.discount_market_price_adjustment)          AS descuento_ajuste_mercado,
    SUM(COALESCE(v.short_dated_discount, 0))         AS descuento_fecha_corta,
    SUM(COALESCE(v.focus_discount, 0))               AS descuento_foco
FROM public.daily_sales_sector_10_15_20_21 v
LEFT JOIN public.materials m
       ON v.material::integer = m.code
LEFT JOIN public.material_groups_6 g6
       ON g6.code = m.gr6
LEFT JOIN public.commercial_distributors_details cdd
       ON v.id_commercial_distributors = cdd.id
LEFT JOIN public.distribution_channels dc
       ON v.distribution_channel = dc.distribution_channel
LEFT JOIN cliente_attr cp
       ON v.client = cp.client
LEFT JOIN public.vw_dim_tipologias_cliente t
       ON t.client_typology_id = LPAD(COALESCE(NULLIF(v.client_typology, ''), cp.tipologia, '0'), 3, '0')
WHERE v.date >= :fecha_inicio
  AND v.date <  :fecha_fin                 -- fin EXCLUSIVO
GROUP BY 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14
HAVING COALESCE(SUM(v.value), 0) <> 0
    OR COALESCE(SUM(v.kilogram), 0) <> 0
ORDER BY fecha, cliente, grupo_material_6;
"""

# Rangos: (limite inferior del % de lista, rotulo), de mayor a menor. El numero del rotulo ordena los tramos.
RANGOS = [(100, "100% O MAS"), (97, "Mas de 97 menos de 100%"), (95, "Mas de 95 menos de 97%"),
          (93, "Mas de 93 menos de 95%"), (90, "Mas de 90 menos de 93%"), (87, "Mas de 87 menos de 90%"),
          (85, "Mas de 85 menos de 87%"), (83, "Mas de 83 menos de 85%"), (80, "Mas de 80 menos de 83%"),
          (77, "Mas de 77 menos de 80%"), (75, "Mas de 75 menos de 77%"), (73, "Mas de 73 menos de 75%"),
          (70, "Mas de 70 menos de 73%"), (67, "Mas de 67 menos de 70%"), (65, "Mas de 65 menos de 67%"),
          (63, "Mas de 63 menos de 65%"), (60, "Mas de 60 menos de 63%"), (57, "Mas de 57 menos de 60%"),
          (55, "Mas de 55 menos de 57%"), (53, "Mas de 53 menos de 55%"), (50, "Mas de 50 menos de 53%")]
_ROTULOS = [f"{i:02d}. {r}" for i, (_, r) in enumerate(RANGOS, start=1)] + [f"{len(RANGOS) + 1:02d}. Menos de 50%"]
_ROTULO_SIN_BRUTA = f"{len(RANGOS) + 2:02d}. Sin venta bruta"
_BORDES = [float("-inf")] + [limite for limite, _ in reversed(RANGOS)] + [float("inf")]
MOTIVOS_FILA = {"descuento_fecha_corta": "Fecha corta", "descuento_ajuste_mercado": "Ajuste mercado",
                "descuento_foco": "Foco"}
COLUMNAS = {
    "fecha": "Fecha", "cliente": "Codigo SAP Cliente", "nombre_cliente": "Nombre Cliente",
    "lista_precio": "Lista de Precios", "venta_kg": "Venta Kg", "venta_neta": "Venta Neta",
    "venta_bruta": "Venta Bruta", "descuento_total": "Valor Total Descuento",
    "descuento_ajuste_mercado": "Descuento Ajuste Precio Mercado",
    "descuento_fecha_corta": "Descuento Rotacion / Fecha Corta", "descuento_foco": "Descuento Foco",
    "pct_descuento_total": "% Descuento Total", "pct_descuento_ajuste": "% Descuento Ajuste Mercado",
    "pct_descuento_fecha_corta": "% Descuento Rotacion / Fecha Corta", "pct_descuento_foco": "% Descuento Foco",
    "precio_promedio": "Precio Promedio", "grupo_material_6": "Grupo Material 6", "nombre_gr6": "Nombre GR6",
    "canal_distribucion": "Canal de Distribucion", "distribuidora": "Distribuidora",
    "nombre_distribuidora": "Nombre Distribuidora", "sociedad": "Sociedad", "tipologia": "Tipologia Cliente",
    "nombre_tipologia": "Nombre Tipologia", "canal_venta": "Canal de Venta", "sector": "Sector",
    "pct_lista": "% Lista Total", "pct_lista_rotacion": "% Lista Dcto Rotacion",
    "pct_lista_ajuste": "% Lista Dcto Ajuste", "pct_lista_foco": "% Lista Dcto Foco",
    "rango": "Rango Total", "rango_rotacion": "Rango Dcto Rotacion", "rango_ajuste": "Rango Dcto Ajuste",
    "rango_foco": "Rango Dcto Foco", "motivo": "Motivo Descuento",
}


def rango(pct_: pd.Series) -> pd.Series:
    """% de lista (0 a 100) -> rotulo del tramo. [97, 100) es 'Mas de 97 menos de 100%'."""
    tramos = pd.cut(pct_, bins=_BORDES, labels=list(reversed(_ROTULOS)), right=False)
    return tramos.astype(str).where(pct_.notna(), _ROTULO_SIN_BRUTA)


def calcular(df: pd.DataFrame) -> pd.DataFrame:
    """Medidas, % sobre la venta bruta, % de lista, rangos y motivo de cada fila."""
    medidas = ["venta_kg", "venta_neta", "venta_bruta", "descuento_total",
               "descuento_ajuste_mercado", "descuento_fecha_corta", "descuento_foco"]
    df[medidas] = df[medidas].astype(float)
    df["precio_promedio"] = (df["venta_neta"] / df["venta_kg"].where(df["venta_kg"] != 0)).round(0)
    bruta = df["venta_bruta"].where(df["venta_bruta"] > 0)   # devoluciones (bruta <= 0): sin %

    def sobre_bruta(serie: pd.Series) -> pd.Series:
        return (serie / bruta * 100).round(2)

    df["pct_descuento_total"] = sobre_bruta(df["descuento_total"])
    df["pct_descuento_ajuste"] = sobre_bruta(df["descuento_ajuste_mercado"])
    df["pct_descuento_fecha_corta"] = sobre_bruta(df["descuento_fecha_corta"])
    df["pct_descuento_foco"] = sobre_bruta(df["descuento_foco"])
    # % de lista: el total usa la neta; los otros aislan un solo descuento, como si fuera el unico
    df["pct_lista"] = sobre_bruta(df["venta_neta"])
    df["pct_lista_rotacion"] = sobre_bruta(df["venta_bruta"] - df["descuento_fecha_corta"])
    df["pct_lista_ajuste"] = sobre_bruta(df["venta_bruta"] - df["descuento_ajuste_mercado"])
    df["pct_lista_foco"] = sobre_bruta(df["venta_bruta"] - df["descuento_foco"])
    for sufijo in ("", "_rotacion", "_ajuste", "_foco"):
        df["rango" + sufijo] = rango(df["pct_lista" + sufijo])
    partes = [df[col].gt(0).map({True: nombre + " + ", False: ""}) for col, nombre in MOTIVOS_FILA.items()]
    texto_ = pd.concat(partes, axis=1).sum(axis=1).str.removesuffix(" + ")
    df["motivo"] = texto_.where(texto_ != "", "Sin descuento")
    df["fecha"] = pd.to_datetime(df["fecha"])
    return df[list(COLUMNAS)].rename(columns=COLUMNAS)


def tipo_parquet(serie: pd.Series) -> pa.DataType:
    if pd.api.types.is_datetime64_any_dtype(serie):
        return pa.timestamp("ms")
    if pd.api.types.is_numeric_dtype(serie):
        return pa.float64()
    return pa.string()


def escribir_parquet(partes, salida: Path) -> int:
    """Escribe parte por parte. El esquema se fija con la primera parte (texto en lo que no es numero ni fecha).
    Si algo falla a mitad, borra el archivo: uno incompleto es peor que ninguno."""
    escritor, filas = None, 0
    try:
        for df in partes:
            if escritor is None:
                esquema = pa.schema([(c, tipo_parquet(df[c])) for c in df.columns])
                escritor = pq.ParquetWriter(salida, esquema)
            escritor.write_table(pa.Table.from_pandas(df, schema=esquema, preserve_index=False))
            filas += len(df)
            print(f"  {filas:,} filas")
    except BaseException:
        if escritor is not None:
            escritor.close()
            salida.unlink(missing_ok=True)
        raise
    if escritor is not None:
        escritor.close()
    return filas

## A.4 Parquet del mes en curso

El SQL de A.3 desde el 1 del mes anterior hasta `CORTE_D` (incluido), para el avance (mes anterior cerrado vs mes
en curso). Si `CORTE_D` es `None`, el corte es ayer. El archivo lleva las fechas en el nombre: un corte nuevo genera uno nuevo. **Si vuelve a correr el mismo
día con el mismo corte, ponga `GENERAR_PARQUET_D = True`** para traer lo que se haya cargado después.

In [ ]:
CORTE_D = CORTE_D or date.today() - timedelta(days=1)          # por defecto, ayer
INICIO_ACT = CORTE_D.replace(day=1)                              # 1 del mes en curso
INICIO_ANT = (INICIO_ACT - timedelta(days=1)).replace(day=1)     # 1 del mes anterior
FIN_D = CORTE_D + timedelta(days=1)                              # EXCLUSIVO
PARQUET_D = AQUI / f"Descuentos_{INICIO_ANT:%Y%m%d}_{FIN_D:%Y%m%d}.parquet"

if GENERAR_PARQUET_D or not PARQUET_D.exists():
    print(f"Consultando {INICIO_ANT} -> {FIN_D} (exclusivo)...")
    with ENGINE.connect() as c:
        c = c.execution_options(stream_results=True)
        partes = pd.read_sql(text(SQL_DESCUENTOS), c, chunksize=FILAS_POR_PARTE,
                             params={"fecha_inicio": INICIO_ANT, "fecha_fin": FIN_D})
        escribir_parquet((calcular(p) for p in partes), PARQUET_D)
print(f"Base del avance: {PARQUET_D.name}")

## A.5 Nombres, formato y funciones comunes

In [ ]:
SOCIEDADES = {"1000": "Región Oriente", "2000": "Región Occidente"}   # company_code: 1000 Región Oriente, 2000 Región Occidente
VISTAS = {"Región Oriente": "Región Oriente", "Región Occidente": "Región Occidente"}
ARCHIVO = {"Región Oriente": "Region_Oriente", "Región Occidente": "Region_Occidente"}   # nombre en los archivos
CORTO = {soc: vista for vista, soc in VISTAS.items()}
NOMBRE_CANAL_VENTA = {"Directo": "Amas de casa"}
CANALES_VENTA_TOP = ["Tradicional", "Institucional", "Modernos", "Amas de casa"]   # orden en tablas y top
ROTULO_CANAL = {"Mayoreo": "Mayoreo", "Puntos Venta": "Puntos de Venta"}   # canal de distribucion
# Dimensiones de los rangos (secciones 3 a 6 y 8): rotulo -> columna del Parquet (None = el total)
DIMENSIONES = {"Total": None,
               "Canal de distribución": "Canal de Distribucion",
               "Canal de venta": "Canal de Venta",
               "Grupo de material 6": "Nombre GR6",
               "Distribuidora": "Nombre Distribuidora"}
# Cruce con el canal de distribucion (seccion 7): rotulo -> columna del Parquet
CRUCES = {"Canal de venta": "Canal de Venta"}
MEDIDAS = ["Venta Kg", "Venta Bruta", "Venta Neta",
           "Descuento Ajuste Precio Mercado", "Descuento Rotacion / Fecha Corta"]

# --- nombres normalizados
VENTA_BRUTA, VENTA_NETA, VENTA_KG = "Venta bruta ($)", "Venta neta ($)", "Venta kg"
PRECIO_LISTA, PRECIO_NETO = "Precio lista por kg", "Precio neto por kg"
L_GEN, D_GEN = "% aplicado al precio de lista general", "% Ajuste general"
L_AJ, D_AJ = "% aplicado al precio de lista ajuste de precio de mercado", "% Ajuste de precio de mercado"
L_FC, D_FC = "% aplicado al precio de lista fecha corta", "% Ajuste de fecha corta"
P_GEN, P_AJ, P_FC = "Ajuste general ($)", "Ajuste de precio de mercado ($)", "Ajuste de fecha corta ($)"
MILLON = 1     # divisor de los montos: 1 = pesos completos (1e6 = millones)

# --- colores: motivo y tipo de ajuste comparten color (ajuste = verde, fecha corta = amarillo)
MOTIVOS = {"Con fecha corta": "#eda100", "Solo ajuste de precio de mercado": "#1baf7a", "Sin ajuste": "#b5b4af"}
COLOR_SOC = {"Región Oriente": "#2a78d6", "Región Occidente": "#eb6834"}

TRAMOS = [r[4:] for r in _ROTULOS]  # "100% O MAS", ..., "Menos de 50%"
MESES_ES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto",
            "septiembre", "octubre", "noviembre", "diciembre"]


# --- formato colombiano
def num(v, dec: int = 1) -> str:
    """Miles con punto, decimales con coma (3.831,3)."""
    if pd.isna(v):
        return "-"
    return f"{v:,.{dec}f}".replace(",", "_").replace(".", ",").replace("_", ".")


def pct(v) -> str:
    return "-" if pd.isna(v) else num(v) + "%"


def pesos(v) -> str:
    return "-" if pd.isna(v) else "$" + num(v, 0)


def variacion(v, en_puntos: bool) -> str:
    """+5,9% (variacion) o -6,4 pts (diferencia entre porcentajes)."""
    if pd.isna(v):
        return "-"
    return ("+" if v >= 0 else "-") + num(abs(v)) + (" pts" if en_puntos else "%")


def formato_de(indicador: str):
    if indicador.startswith("%"):
        return pct
    if indicador.startswith("Venta kg"):   # kilos completos, sin decimales
        return lambda v: num(v, 0)
    if indicador.startswith("Precio") or indicador.endswith("($)"):
        return pesos
    return num


# --- indicadores (todo desde sumas, nunca promediando % de filas)
def indicadores(s: pd.DataFrame) -> pd.DataFrame:
    """s: sumas de MEDIDAS (cualquier indice) -> BLOQUE de indicadores."""
    bruta = s["Venta Bruta"]
    b = bruta.where(bruta != 0)
    kg = s["Venta Kg"].where(s["Venta Kg"] != 0)
    aj, fc = s["Descuento Ajuste Precio Mercado"], s["Descuento Rotacion / Fecha Corta"]
    gen = aj + fc
    return pd.DataFrame({
        VENTA_BRUTA: bruta / MILLON,
        VENTA_NETA: s["Venta Neta"] / MILLON,
        VENTA_KG: s["Venta Kg"],
        PRECIO_LISTA: bruta / kg,
        PRECIO_NETO: s["Venta Neta"] / kg,
        L_GEN: (bruta - gen) / b * 100,
        D_GEN: gen / b * 100,
        P_GEN: gen / MILLON,
        L_AJ: (bruta - aj) / b * 100,
        D_AJ: aj / b * 100,
        P_AJ: aj / MILLON,
        L_FC: (bruta - fc) / b * 100,
        D_FC: fc / b * 100,
        P_FC: fc / MILLON,
    }, index=s.index)


def comparar(t: pd.DataFrame) -> pd.DataFrame:
    """t: filas = años -> filas = indicadores; columnas 2025, 2026 y Variación (% o pts)."""
    t = t.T
    out = t[[A1, A2]].copy()
    out.columns = [str(A1), str(A2)]
    es_pct = out.index.str.startswith("%")
    out["Variación"] = ((out[str(A2)] / out[str(A1)] - 1) * 100).where(~es_pct, out[str(A2)] - out[str(A1)])
    return out


def es_variacion(col) -> bool:
    """Columna de variación entre años: 'Variación' en cualquier nivel del encabezado."""
    return any(str(x).startswith("Variación") for x in (col if isinstance(col, tuple) else (col,)))


def ejes_limpios(ax, formato_y=None, formato_x=None):
    ax.grid(axis="y" if formato_y else "x", color="#e6e5e1", linewidth=0.8)
    ax.set_axisbelow(True)
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)
    ax.tick_params(colors="#52514e", labelsize=8)
    if formato_y:
        ax.yaxis.set_major_formatter(formato_y)
    if formato_x:
        ax.xaxis.set_major_formatter(formato_x)

## A.6 Lectura del Parquet del mes

Una pasada por lotes que deja tablas chicas con el periodo (mes anterior / mes en curso) en el nivel `Año`:
`diario_soc` (sociedad × día), `cubo_d`, `cruce_d`, `clientes_d`, `foco_d` (los materiales de `FOCO` por canal de
venta, canal de distribución, distribuidora, cliente, rango y motivo) y `foco_dia_d` (los materiales de `FOCO` juntos
por sociedad, canal de venta y día). La sociedad sale de `company_code`: si el Parquet no trae la columna
`Sociedad`, se asigna por distribuidora.

In [ ]:
archivo_d = pq.ParquetFile(PARQUET_D)
i_fecha = archivo_d.schema_arrow.get_field_index("Fecha")
FECHA_MAX_D = pd.Timestamp(max(archivo_d.metadata.row_group(i).column(i_fecha).statistics.max
                               for i in range(archivo_d.metadata.num_row_groups)))
MES_ANT_TXT, MES_ACT_TXT = MESES_ES[INICIO_ANT.month - 1], MESES_ES[INICIO_ACT.month - 1]
ETQ_ANT = f"{MES_ANT_TXT[:3].capitalize()} {INICIO_ANT.year}"          # "Sep 2026"
ETQ_ACT = f"{MES_ACT_TXT[:3].capitalize()} al {FECHA_MAX_D.day}"       # "Oct al 13"
PERIODOS_D = [ETQ_ANT, ETQ_ACT]

columnas = ["Fecha", "Distribuidora", *MEDIDAS, "Rango Total", "Codigo SAP Cliente", "Nombre Cliente",
            *[c for c in DIMENSIONES.values() if c]]


def juntar(partes: list) -> pd.DataFrame:
    t = pd.concat(partes)
    return t.groupby(level=list(range(t.index.nlevels))).sum()


con_sociedad = "Sociedad" in archivo_d.schema_arrow.names
columnas_d = [c for c in columnas if c != "Sociedad"] + (["Sociedad"] if con_sociedad else [])
if not con_sociedad:
    with ENGINE.connect() as c:
        sociedad_de = pd.read_sql(text(
            "SELECT DISTINCT new_zone_code, company_code::text AS sociedad "
            "FROM public.commercial_distributors_details"), c).set_index("new_zone_code")["sociedad"]
    assert sociedad_de.index.is_unique, "Hay distribuidoras en mas de una sociedad"

partes_dia, partes_cubo, partes_cruce, partes_cli, nombres, partes_cv, partes_foco = [], [], [], [], [], [], []
partes_foco_dia = []
for lote in archivo_d.iter_batches(batch_size=FILAS_POR_PARTE, columns=columnas_d):
    d = lote.to_pandas()
    if not con_sociedad:
        d["Sociedad"] = d["Distribuidora"].map(sociedad_de)
    d["Sociedad"] = d["Sociedad"].map(SOCIEDADES).fillna(d["Sociedad"]).fillna("Sin sociedad")
    d = d[d["Sociedad"].isin(VISTAS.values())]
    d["Canal de Venta"] = d["Canal de Venta"].replace(NOMBRE_CANAL_VENTA)
    d["Dia"] = d["Fecha"].dt.to_period("D")
    partes_dia.append(d.groupby(["Sociedad", "Dia"])[MEDIDAS].sum())

    # el nivel "Año" lleva el periodo: mes anterior cerrado o mes en curso
    d["Año"] = pd.Series(ETQ_ANT, index=d.index).mask(d["Fecha"] >= pd.Timestamp(INICIO_ACT), ETQ_ACT)
    d["Motivo"] = (pd.Series("Sin ajuste", index=d.index)
                   .mask(d["Descuento Ajuste Precio Mercado"] > 0, "Solo ajuste de precio de mercado")
                   .mask(d["Descuento Rotacion / Fecha Corta"] > 0, "Con fecha corta"))
    for dim, col in DIMENSIONES.items():
        miembro = (d[col].fillna("(vacío)") if col else pd.Series("Total", index=d.index)).rename("Miembro")
        partes_cubo.append(pd.concat(
            {dim: d.groupby([miembro, "Sociedad", "Año", "Rango Total", "Motivo"])[MEDIDAS].sum()},
            names=["Dimension"]))
    for dim, col in CRUCES.items():
        partes_cruce.append(pd.concat(
            {dim: d.groupby([d[col].fillna("(vacío)").rename("Miembro"), "Canal de Distribucion",
                             "Sociedad", "Año"])[MEDIDAS].sum()}, names=["Dimension"]))
    partes_cli.append(d.groupby(["Codigo SAP Cliente", "Sociedad", "Canal de Distribucion", "Año"])[MEDIDAS].sum())
    nombres.append(d.groupby("Codigo SAP Cliente")["Nombre Cliente"].first())
    partes_cv.append(d[d["Año"] == ETQ_ACT].groupby(["Codigo SAP Cliente", "Canal de Venta"])["Venta Bruta"].sum())
    f = d[d["Nombre GR6"].isin(FOCO)].fillna({"Canal de Venta": "(sin canal)", "Nombre Distribuidora": "(vacío)",
                                               "Canal de Distribucion": "(vacío)"})
    partes_foco.append(f.groupby(["Nombre GR6", "Sociedad", "Año", "Canal de Venta", "Nombre Distribuidora",
                                  "Canal de Distribucion", "Codigo SAP Cliente", "Rango Total", "Motivo"])[MEDIDAS].sum())
    partes_foco_dia.append(f.groupby(["Sociedad", "Canal de Venta", "Dia"])[MEDIDAS].sum())

diario_soc = juntar(partes_dia)           # sociedad x dia
cubo_d = juntar(partes_cubo)
cruce_d = juntar(partes_cruce)
clientes_d = juntar(partes_cli)           # cliente x sociedad x canal de distribucion x periodo
foco_d = juntar(partes_foco)              # material x sociedad x periodo x canales x distribuidora x cliente x rango
                                          # x motivo
foco_dia_d = juntar(partes_foco_dia)      # materiales de FOCO juntos: sociedad x canal de venta x dia
nombre_cliente_d = pd.concat(nombres)
nombre_cliente_d = nombre_cliente_d[~nombre_cliente_d.index.duplicated()]
canal_venta_cliente_d = (pd.concat(partes_cv).groupby(level=[0, 1]).sum().reset_index()
                         .sort_values("Venta Bruta").drop_duplicates("Codigo SAP Cliente", keep="last")
                         .set_index("Codigo SAP Cliente")["Canal de Venta"])
del partes_dia, partes_cubo, partes_cruce, partes_cli, nombres, partes_cv, partes_foco, partes_foco_dia

faltan = [m for m in FOCO if m not in foco_d.index.unique("Nombre GR6")]
assert not faltan, f"Sin venta en el periodo para: {faltan} (revise el nombre en FOCO)"
print(f"Datos hasta {FECHA_MAX_D:%d/%m/%Y} · {ETQ_ANT} vs {ETQ_ACT} | cubo {len(cubo_d):,} filas · "
      f"clientes {len(clientes_d):,} filas · foco {len(foco_d):,} filas")

# B. Funciones de las láminas

Solo las definiciones que usan los PowerPoint, con la numeración de secciones del informe general. Todas corren
dentro de `modo_corte()` (sección 10), que pone los periodos y las tablas del mes anterior / mes en curso. Los % se
calculan siempre desde las sumas (nunca promediando % de filas).

## 3. Rangos de % aplicado al precio de lista: definiciones

Solo las definiciones de esta sección que usa este notebook; la sección completa está en `Informe_Ajustes_Region.ipynb`.

In [ ]:
def con_rango(c: pd.DataFrame) -> pd.DataFrame:
    """Solo las ventas con venta bruta positiva (las que tienen rango)."""
    return c[c.index.get_level_values("Rango Total") != _ROTULO_SIN_BRUTA]


def rango_principal(c: pd.DataFrame, anio: int):
    """Rango con mas venta bruta entre las ventas CON ajuste (sin el rango 100% o mas). None si no hay venta."""
    r = con_rango(c)
    r = r[(r.index.get_level_values("Año") == anio)
          & (r.index.get_level_values("Rango Total") != _ROTULOS[0])]["Venta Bruta"].groupby(level="Rango Total").sum()
    r = r[r > 0]
    return r.idxmax()[4:] if len(r) else None


def frase_rango(c: pd.DataFrame) -> str:
    """'la venta con ajuste pasó del rango X al Y' / 'sigue concentrada en X'."""
    r1, r2 = rango_principal(c, A1), rango_principal(c, A2)
    if r2 is None:
        return f"sin venta con ajuste en {A2}"
    if r1 is None:
        return f"la venta con ajuste se concentra en “{r2}” (sin venta en {A1})"
    if r1 == r2:
        return f"la venta con ajuste sigue concentrada en “{r2}”"
    return f"la venta con ajuste pasó del rango “{r1}” al “{r2}”"


SIGLA_MOTIVO = {"Con fecha corta": "FC", "Solo ajuste de precio de mercado": "AM", "Sin ajuste": "SA"}


def figura_rangos(c: pd.DataFrame, titulo: str, figsize=(13, 7.4), detalle=False):
    """22 rangos (filas) x motivo, un panel por año. Largo = % de la venta bruta del año; al lado, % y ajuste general.
    detalle: al final de la etiqueta, el % de la venta bruta de cada motivo (siglas de SIGLA_MOTIVO, en la leyenda)."""
    cr = con_rango(c)
    r = cr["Venta Bruta"]
    por_motivo = r.groupby(level=["Año", "Rango Total", "Motivo"]).sum()
    dcto = cr["Descuento Ajuste Precio Mercado"] + cr["Descuento Rotacion / Fecha Corta"]
    por_rango = dcto.groupby(level=["Año", "Rango Total"]).sum()
    total_anio = r.groupby(level="Año").sum()
    fig, ejes = plt.subplots(1, len(ANIOS), figsize=figsize, sharey=True, sharex=True)
    tope = 0
    for ax, anio in zip(ejes, ANIOS):
        if anio not in total_anio.index:
            ax.set_title(f"{anio}: sin venta", loc="left", fontsize=11)
            continue
        t = (por_motivo[por_motivo.index.get_level_values("Año") == anio].droplevel("Año").unstack("Motivo")
             .reindex(_ROTULOS).fillna(0) / total_anio[anio] * 100)
        pesos_ = por_rango[por_rango.index.get_level_values("Año") == anio].droplevel("Año").reindex(_ROTULOS).fillna(0)
        base = pd.Series(0.0, index=t.index)
        for m, color in MOTIVOS.items():
            v = t[m] if m in t else pd.Series(0.0, index=t.index)
            ax.barh(TRAMOS, v, left=base, color=color, edgecolor="white", linewidth=1, height=0.75,
                    label=f"{m} ({SIGLA_MOTIVO[m]})" if detalle else m)
            base += v
        for i, (total_, plata) in enumerate(zip(base, pesos_)):
            if total_ >= 0.1:
                etiqueta = f"{pct(total_)} · ajuste {pesos(plata)}"
                if detalle:   # solo los motivos que se alcanzan a ver (0,1% o mas)
                    etiqueta += "".join(f" · {s_} {pct(t[m].iloc[i])}" for m, s_ in SIGLA_MOTIVO.items()
                                        if m in t and t[m].iloc[i] >= 0.05)
                ax.annotate(etiqueta, (total_, i), xytext=(3, 0),
                            textcoords="offset points", va="center", fontsize=6.5, color="#52514e")
        tope = max(tope, base.max())
        ax.set_title(str(anio), loc="left", fontsize=11)
        ejes_limpios(ax, formato_x=lambda v, _: f"{v:,.0f}%")
    for ax in ejes:
        ax.set_xlim(0, tope * (3.0 if detalle else 2.1))  # espacio para la etiqueta del ajuste
    ejes[0].invert_yaxis()
    ejes[-1].legend(loc="lower right", frameon=False, fontsize=8)
    fig.suptitle(titulo, x=0.01, ha="left", fontsize=11)
    plt.tight_layout()
    return fig

## 8. Concentración del ajuste – Distribuidoras: definiciones

Solo las definiciones de esta sección que usa este notebook; la sección completa está en `Informe_Ajustes_Region.ipynb`.

In [ ]:
def pareto(s: pd.DataFrame, nombre: str) -> pd.DataFrame:
    """s: sumas por (miembro, año). Ordena por ajuste general del ultimo año y acumula."""
    ind = indicadores(s)
    w2 = ind[ind.index.get_level_values("Año") == A2].droplevel("Año").sort_values(P_GEN, ascending=False)
    w1 = ind[ind.index.get_level_values("Año") == A1].droplevel("Año").reindex(w2.index)
    a1, a2 = str(A1), str(A2)
    out = pd.DataFrame({
        (VENTA_KG, a1): w1[VENTA_KG], (VENTA_KG, a2): w2[VENTA_KG],
        (VENTA_BRUTA, a2): w2[VENTA_BRUTA], (P_GEN, a1): w1[P_GEN], (P_GEN, a2): w2[P_GEN],
        (VENTA_NETA, a2): w2[VENTA_NETA],
        ("% acumulado ajuste general", a2): w2[P_GEN].clip(lower=0).cumsum() / w2[P_GEN].clip(lower=0).sum() * 100,
        (L_GEN, a1): w1[L_GEN], (L_GEN, a2): w2[L_GEN], (D_GEN, a2): w2[D_GEN],
        (L_AJ, a2): w2[L_AJ], (D_AJ, a2): w2[D_AJ],
        (L_FC, a1): w1[L_FC], (L_FC, a2): w2[L_FC], (D_FC, a1): w1[D_FC], (D_FC, a2): w2[D_FC],
    })
    out.index.name = nombre
    return out

## 9. Concentración del ajuste – Clientes Mayoreo / Puntos de Venta: definiciones

Solo las definiciones de esta sección que usa este notebook; la sección completa está en `Informe_Ajustes_Region.ipynb`.

In [ ]:
COLS_TOP = ["Canal de venta", "Distribuidora", VENTA_KG, VENTA_BRUTA, P_GEN, D_GEN, P_AJ, D_AJ, P_FC, D_FC]


def principal_top(k: pd.DataFrame, nivel: str) -> pd.Series:
    """Valor del nivel (canal de venta / distribuidora) con mas venta bruta para cada cliente."""
    x = k["Venta Bruta"].groupby(level=["Codigo SAP Cliente", nivel]).sum().reset_index()
    x = x.sort_values("Venta Bruta").drop_duplicates("Codigo SAP Cliente", keep="last")
    return x.set_index("Codigo SAP Cliente")[nivel]


def top_de(k: pd.DataFrame, n=TOP_CLIENTES):
    """Top n clientes por ajuste general ($). Devuelve (tabla, % del ajuste general que suman, total, clientes con ajuste)."""
    g = k.groupby(level="Codigo SAP Cliente")[MEDIDAS].sum()
    gen = g["Descuento Ajuste Precio Mercado"] + g["Descuento Rotacion / Fecha Corta"]
    g, gen = g[gen > 0], gen[gen > 0]
    if g.empty:
        return None, 0, 0, 0
    total, n_cli = gen.sum(), len(g)
    orden = gen.sort_values(ascending=False).head(n).index
    g, gen = g.loc[orden], gen.loc[orden]
    ind = indicadores(g)
    canal, dist = principal_top(k, "Canal de Venta"), principal_top(k, "Nombre Distribuidora")
    t = pd.DataFrame({
        "Canal de venta": [canal.get(c_, "-") for c_ in g.index],
        "Distribuidora": [str(dist.get(c_, "-")).title() for c_ in g.index],
        VENTA_KG: ind[VENTA_KG],
        VENTA_BRUTA: g["Venta Bruta"],
        P_GEN: gen,
        D_GEN: ind[D_GEN],
        P_AJ: g["Descuento Ajuste Precio Mercado"],
        D_AJ: ind[D_AJ],
        P_FC: g["Descuento Rotacion / Fecha Corta"],
        D_FC: ind[D_FC],
    }, index=g.index)[COLS_TOP]
    t.index = [f"{cod} - {nombre_cliente.get(cod, '')}" for cod in t.index]
    t.index.name = "Cliente"
    return t, gen.sum() / total * 100, total, n_cli

## 10. Avance del mes en curso: definiciones

Solo las definiciones de esta sección que usa este notebook; la sección completa está en `Informe_Ajustes_Region.ipynb`.

In [ ]:
DIAS_EJE = pd.period_range(INICIO_ANT, FECHA_MAX_D, freq="D")
COLS_FOCO = [VENTA_KG, P_GEN, PRECIO_NETO, L_GEN, D_AJ, D_FC]
KG_DIA, BRUTA_DIA = "Venta kg por día", "Venta bruta por día ($)"


def diario_de(soc) -> pd.DataFrame:
    """Sumas por dia de la sociedad (en el modo de corte reemplaza a mensual_de)."""
    return diario_soc[diario_soc.index.get_level_values("Sociedad") == soc].groupby(level="Dia").sum()


def dias_con_venta(soc) -> pd.Series:
    """Dias con venta bruta en cada periodo."""
    dd = diario_de(soc)
    dd = dd[dd["Venta Bruta"] > 0]
    return pd.Series(dd.index.to_timestamp() >= pd.Timestamp(INICIO_ACT), index=dd.index).map(
        {False: ETQ_ANT, True: ETQ_ACT}).value_counts().reindex(PERIODOS_D, fill_value=0)


def panorama_corte(soc) -> pd.DataFrame:
    """La tabla del panorama (A1 = mes anterior, A2 = mes en curso) mas la venta por dia con venta."""
    s = cubo_d.xs("Total", level="Dimension")
    s = s[s.index.get_level_values("Sociedad") == soc].groupby(level="Año").sum().reindex(PERIODOS_D)
    ind = indicadores(s)
    dias = dias_con_venta(soc)
    ind[KG_DIA] = ind[VENTA_KG] / dias
    ind[BRUTA_DIA] = ind[VENTA_BRUTA] / dias
    return comparar(ind)


CAMBIOS_CORTE = {
    "A1": ETQ_ANT, "A2": ETQ_ACT, "ANIOS": PERIODOS_D, "PERIODO": "mes cerrado vs en curso ·",
    "MES_FIN_TXT": f"{MES_ACT_TXT} al {FECHA_MAX_D.day}", "MES_TXT": ETQ_ACT, "FECHA_MAX": FECHA_MAX_D,
    "cubo": cubo_d, "cruce": cruce_d, "clientes": clientes_d, "nombre_cliente": nombre_cliente_d,
    "canal_venta_cliente": canal_venta_cliente_d,
    "CANALES": sorted(clientes_d.index.unique("Canal de Distribucion")),
    "CANALES_DIST": sorted(cruce_d.index.unique("Canal de Distribucion")),
    # evolucion dia a dia (seccion 2)
    "mensual_de": diario_de, "MESES_EJE": DIAS_EJE, "X_MES": list(range(len(DIAS_EJE))),
    "ETIQ_MES": DIAS_EJE.strftime("%d/%m"), "CORTE_ANIO": sum(DIAS_EJE.month == INICIO_ANT.month) - 0.5,
    "PANELES": {"kg": "Venta kg por día",
                "pesos": "Venta bruta y venta neta por día ($) — la franja es el ajuste",
                "precio": "Precio de lista y precio neto por kg, por día",
                "aplicado": "% aplicado al precio de lista por día (100% − % de ajuste)"},
    # rotulos del informe general (C.2) que llevan los años
    "ANIOS_CONC": {L_GEN: (ETQ_ANT, ETQ_ACT), D_FC: (ETQ_ANT, ETQ_ACT), L_FC: (ETQ_ANT, ETQ_ACT)},
    "PANORAMA": None,   # se calcula al entrar
}


@contextmanager
def modo_corte():
    """Mientras dura el `with`, las funciones de B y C comparan el mes anterior cerrado con el mes en curso; al salir
    (aunque haya error) todo queda como estaba. Los rotulos de las secciones (C.2) se toman si ya existen."""
    g = globals()
    cambios = {**CAMBIOS_CORTE, "SECCIONES": {**g.get("SECCIONES", {}), 2: f"Panorama {ETQ_ANT} vs {ETQ_ACT}"}}
    antes = {k: g[k] for k in cambios if k in g}
    g.update(cambios)
    try:
        g["PANORAMA"] = {vista: panorama_corte(soc) for vista, soc in VISTAS.items()}
        yield
    finally:
        for k in cambios:
            if k in antes:
                g[k] = antes[k]
            else:
                g.pop(k, None)

## 11. Foco Res: definiciones

Las tablas de los materiales de `FOCO` en el canal de venta. `solo_canal()` deja en `foco_d` solo un canal de venta
mientras dura el `with`.

In [ ]:
def foco_de(soc, material, periodo=None) -> pd.DataFrame:
    """Filas de foco_d de un material en la sociedad (y en un periodo, si se da)."""
    f = foco_d[(foco_d.index.get_level_values("Sociedad") == soc)
               & (foco_d.index.get_level_values("Nombre GR6") == material)]
    if periodo is not None:
        f = f[f.index.get_level_values("Año") == periodo]
    return f


def a_columnas(filas: dict, nombre: str) -> pd.DataFrame:
    """{rotulo: indicadores por periodo} -> rotulo x (indicador, periodo) con COLS_FOCO, en el orden del dict."""
    t = pd.concat({k: v[COLS_FOCO].reindex(ANIOS) for k, v in filas.items()}, names=[nombre, "Año"])
    t = t.unstack("Año").reindex(columns=pd.MultiIndex.from_product([COLS_FOCO, ANIOS])).reindex(list(filas))
    t.index.name = nombre
    return t


@contextmanager
def solo_canal(canal_venta):
    """Mientras dura el `with`, foco_d (el vigente: mes o acumulado) solo trae el canal de venta dado."""
    g = globals()
    antes = g["foco_d"]
    g["foco_d"] = antes[antes.index.get_level_values("Canal de Venta") == canal_venta]
    try:
        yield
    finally:
        g["foco_d"] = antes


def canales_dist_foco(soc, m) -> list:
    """Canales de distribucion con venta del material (Mayoreo, Puntos de Venta y luego los demas)."""
    b = foco_de(soc, m)["Venta Bruta"].groupby(level="Canal de Distribucion").sum()
    cds = b[b > 0].index
    return [c for c in ROTULO_CANAL if c in cds] + sorted(c for c in cds if c not in ROTULO_CANAL)


def foco_canal_cd(soc, m) -> pd.DataFrame:
    """Material en el canal de venta vigente: cada canal de distribucion y el total del canal (x indicador, periodo)."""
    f = foco_de(soc, m)
    cd = indicadores(f.groupby(level=["Canal de Distribucion", "Año"]).sum())
    filas = {ROTULO_CANAL.get(c, c): cd.xs(c, level="Canal de Distribucion") for c in canales_dist_foco(soc, m)}
    filas["Total"] = indicadores(f.groupby(level="Año").sum())
    return a_columnas(filas, m)


def foco_distribuidoras(soc, material) -> pd.DataFrame:
    """Pareto de distribuidoras del material, ordenadas por ajuste general del mes en curso."""
    s = foco_de(soc, material).groupby(level=["Nombre Distribuidora", "Año"]).sum()
    return pareto(s.rename_axis(["Miembro", "Año"]), "Distribuidora")


def foco_clientes(soc, material):
    """Top de clientes del material en el ultimo periodo (como C.3), con su canal de distribucion y el % de ajuste
    general del periodo anterior. Devuelve (tabla, % del ajuste general del material que suman, ajuste general,
    clientes con ajuste)."""
    k = foco_de(soc, material, A2)
    t, part, total, n_cli = top_de(k)
    if t is None:
        return None, 0, 0, 0
    codigos = [i.split(" - ", 1)[0] for i in t.index]
    canal = principal_top(k, "Canal de Distribucion")
    t.insert(1, "Canal de distribución", [ROTULO_CANAL.get(canal.get(c_), canal.get(c_, "-")) for c_ in codigos])
    ant = indicadores(foco_de(soc, material, A1).groupby(level="Codigo SAP Cliente")[MEDIDAS].sum())[D_GEN]
    t.insert(t.columns.get_loc(D_GEN) + 1, f"{D_GEN} {A1}", [ant.get(c_, float("nan")) for c_ in codigos])
    return t, part, total, n_cli


def calor_ajustes(t: pd.DataFrame, indice) -> pd.DataFrame:
    """Mapa de calor solo en las columnas de % de ajuste (1 o 2 niveles de encabezado)."""
    es_aj = [(c[0] if isinstance(c, tuple) else c).startswith("% Ajuste") for c in t.columns]
    calor = t[[c for c, s in zip(t.columns, es_aj) if s]].reindex(columns=t.columns)
    calor.index = indice
    return calor

# C. Salidas

Requieren las partes A y B: aquí solo se generan los archivos, en `SALIDA`:

| Sección | Archivo |
|---|---|
| C.7 Res | `Res_<región>_<canal de venta>_<mes anterior>_vs_<mes en curso>_al_<día>.pptx` (8 archivos) |

## C.1 Funciones comunes de PowerPoint

In [ ]:
from pptx import Presentation
from pptx.dml.color import RGBColor
from pptx.enum.text import MSO_ANCHOR, PP_ALIGN
from pptx.util import Inches, Pt


def rgb(h: str) -> RGBColor:
    return RGBColor.from_string(h.lstrip("#").upper())


TINTA, TINTA_2, TINTA_3 = rgb("#0b0b0b"), rgb("#52514e"), rgb("#8c8b87")
FONDO_ENC, BLANCO, ACENTO = rgb("#f0efeb"), rgb("#ffffff"), rgb("#2a78d6")
ANCHO, ALTO, MARGEN = Inches(13.333), Inches(7.5), Inches(0.5)
UTIL_W = ANCHO - 2 * MARGEN
CUERPO_Y = Inches(1.6)
CUERPO_H = ALTO - CUERPO_Y - Inches(0.55)
prs = None   # presentacion en construccion


def nueva_presentacion():
    global prs
    prs = Presentation()
    prs.slide_width, prs.slide_height = ANCHO, ALTO


def guardar(nombre: str):
    archivo_ = SALIDA / nombre
    try:
        prs.save(archivo_)
        print(f"{archivo_.name}: {len(prs.slides)} láminas")
    except PermissionError:
        print(f"No se pudo guardar {archivo_.name}: ciérrelo en PowerPoint y vuelva a correr esta celda.")


def caja_texto(d, x, y, w, h, parrafos, tam=14, color=TINTA, negrita=False, alinear=PP_ALIGN.LEFT, espacio=0):
    """parrafos: texto o lista; cada parrafo es texto o lista de (texto, negrita)."""
    caja = d.shapes.add_textbox(x, y, w, h)
    tf = caja.text_frame
    tf.word_wrap = True
    for i, par in enumerate([parrafos] if isinstance(parrafos, str) else parrafos):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.alignment = alinear
        p.space_after = Pt(espacio)
        for trozo, neg in ([(par, negrita)] if isinstance(par, str) else par):
            r = p.add_run()
            r.text = trozo
            r.font.size, r.font.bold = Pt(tam), neg
            r.font.color.rgb = color
    return caja


def portada(titulo: str, linea_1: str, linea_2: str, color: str):
    """Titulo, barra de color, dos lineas de contexto y el area al pie."""
    d = prs.slides.add_slide(prs.slide_layouts[6])
    caja_texto(d, MARGEN, Inches(2.2), UTIL_W, Inches(0.9), titulo, tam=28, negrita=True)
    barra = d.shapes.add_shape(1, MARGEN, Inches(3.2), Inches(1.2), Inches(0.06))
    barra.fill.solid()
    barra.fill.fore_color.rgb = rgb(color)
    barra.line.fill.background()
    caja_texto(d, MARGEN, Inches(3.4), UTIL_W, Inches(0.4), linea_1, tam=18, color=TINTA_2)
    caja_texto(d, MARGEN, Inches(4.0), UTIL_W, Inches(0.34), linea_2, tam=14, color=TINTA_2)
    caja_texto(d, MARGEN, Inches(6.5), UTIL_W, Inches(0.4), "Sistemas de Información", tam=12, color=TINTA_3)


def nombre_sociedad(soc: str) -> str:
    """Nombre de la region para las portadas."""
    return soc


def imagen(d, fig, x=MARGEN, y=CUERPO_Y, w=UTIL_W, h=CUERPO_H):
    """Figura ajustada a la caja w x h, sin deformarla y centrada en horizontal."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=170, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    buf.seek(0)
    pic = d.shapes.add_picture(buf, x, y)
    escala = min(w / pic.width, h / pic.height)
    pic.width, pic.height = int(pic.width * escala), int(pic.height * escala)
    pic.left = int(x + (w - pic.width) / 2)
    return pic


def color_calor(k: float) -> RGBColor:
    """Escala de un solo tono: blanco (k=0) a naranja (k=1), con texto oscuro legible en todo el rango."""
    a, b = (255, 255, 255), (0xE0, 0x6A, 0x2E)
    k = 0.06 + 0.7 * max(0.0, min(1.0, k))
    return RGBColor(*[int(x + (y - x) * k) for x, y in zip(a, b)])


def tabla(d, df: pd.DataFrame, x=MARGEN, y=CUERPO_Y, w=UTIL_W, tam=10, ancho_indice=0.25,
          negrita_filas=(), alto_fila=None, calor=None, anchos=None):
    """Tabla nativa con el texto ya formateado. Columnas de 2 niveles -> 2 filas de encabezado (combinadas).
    calor: DataFrame numerico del mismo tamaño; las columnas con valores se pintan como mapa de calor. La escala es
    por columna, o por indicador (todos sus años juntos) si las columnas tienen 2 niveles.
    anchos: ancho de cada columna en pulgadas, incluida la del indice (si no, se reparten por igual)."""
    if calor is not None:
        calor = calor.astype(float).copy()
        grupos = (calor.columns.get_level_values(0) if isinstance(calor.columns, pd.MultiIndex)
                  else calor.columns)
        for g in pd.unique(grupos):
            cols_g = [col for col, gg in zip(calor.columns, grupos) if gg == g]
            bloque = calor[cols_g]
            lo, hi = bloque.min().min(), bloque.max().max()
            calor[cols_g] = (bloque - lo) / ((hi - lo) or 1)
    dos = isinstance(df.columns, pd.MultiIndex)
    n_enc = 2 if dos else 1
    n_f, n_c = df.shape[0] + n_enc, df.shape[1] + 1
    alto_fila = alto_fila or Inches(0.08 + tam * 0.021)
    forma = d.shapes.add_table(n_f, n_c, x, y, w, alto_fila * n_f)
    t = forma.table
    t.first_row, t.horz_banding = True, False
    t.columns[0].width = int(w * ancho_indice)
    for j in range(1, n_c):
        t.columns[j].width = int(w * (1 - ancho_indice) / (n_c - 1))
    for j, a_ in enumerate(anchos or []):
        t.columns[j].width = Inches(a_)
    for i in range(n_f):
        t.rows[i].height = alto_fila
        for j in range(n_c):
            celda_ = t.cell(i, j)
            if i < n_enc:
                if j == 0:
                    texto_ = (df.index.name or "") if i == n_enc - 1 else ""
                elif dos:
                    texto_ = str(df.columns[j - 1][i])
                else:
                    texto_ = str(df.columns[j - 1])
            else:
                texto_ = str(df.index[i - n_enc]) if j == 0 else str(df.iat[i - n_enc, j - 1])
            celda_.text = texto_
            celda_.fill.solid()
            celda_.fill.fore_color.rgb = FONDO_ENC if i < n_enc else BLANCO
            if calor is not None and i >= n_enc and j > 0 and pd.notna(calor.iat[i - n_enc, j - 1]):
                celda_.fill.fore_color.rgb = color_calor(calor.iat[i - n_enc, j - 1])
            celda_.vertical_anchor = MSO_ANCHOR.MIDDLE
            celda_.margin_left = celda_.margin_right = Inches(0.04)
            celda_.margin_top = celda_.margin_bottom = Inches(0.01)
            p = celda_.text_frame.paragraphs[0]
            p.alignment = PP_ALIGN.LEFT if j == 0 else (PP_ALIGN.CENTER if i < n_enc else PP_ALIGN.RIGHT)
            for r in p.runs:
                r.font.size = Pt(tam)
                r.font.bold = i < n_enc or df.index[i - n_enc] in negrita_filas
                r.font.color.rgb = TINTA
    if dos:  # combina el encabezado superior repetido
        j = 1
        while j < n_c:
            k = j
            while k + 1 < n_c and df.columns[k][0] == df.columns[j - 1][0]:
                k += 1
            if k > j:
                for jj in range(j + 1, k + 1):   # vaciar antes de combinar: merge junta los textos
                    t.cell(0, jj).text = ""
                t.cell(0, j).merge(t.cell(0, k))
            j = k + 1
    return forma


def a_texto(t: pd.DataFrame, filas_indicador=False) -> pd.DataFrame:
    """Formatea una tabla: por fila (indicadores en filas) o por columna (nivel 0)."""
    out = t.astype(object).copy()
    for c in t.columns:
        for i in t.index:
            ind = i if filas_indicador else (c[0] if isinstance(c, tuple) else c)
            v = t.at[i, c]
            if isinstance(v, str):          # columnas de texto (p. ej. canal de venta)
                continue
            out.at[i, c] = variacion(v, ind.startswith("%")) if es_variacion(c) else formato_de(ind)(v)
    return out


def corto(ind: str) -> str:
    """Rotulo corto para encabezados de tabla."""
    return ind.replace("% aplicado al precio de lista", "% aplicado").replace("de precio de mercado", "precio mercado")

## C.2 Informe general por sociedad: definiciones

Solo las definiciones de esta sección que usa este notebook; la sección completa está en `Informe_Ajustes_Region.ipynb`.

In [ ]:
COLS_CONC = [P_GEN, "% acumulado ajuste general", VENTA_KG, VENTA_BRUTA, L_GEN, D_GEN, L_AJ, D_AJ, L_FC, D_FC]
ANCHOS_CONC = [1.51, 1.15, 0.65, 0.76, 1.01] + [0.81] * 9   # pulgadas, como en el informe revisado


def lamina(titulo: str = None, subtitulo: str = None, seccion: str = None, compacta=False):
    """Lamina del informe general: seccion (azul, 16), titular (de 19 a 14 segun el largo, para que quepa en una
    linea), bajada y numero de lamina. compacta: todo mas arriba, para tablas largas."""
    d = prs.slides.add_slide(prs.slide_layouts[6])
    y_sec, y_tit, y_sub = (0.08, 0.41, 0.75) if compacta else (0.25, 0.62, 1.13)
    caja_texto(d, MARGEN, Inches(y_sec), UTIL_W, Inches(0.37), seccion.upper(), tam=16, color=ACENTO, negrita=True)
    if titulo:
        tam = next((t_ for t_ in (19, 16, 14) if len(titulo) <= 1770 / t_), 14)
        caja_texto(d, MARGEN, Inches(y_tit), UTIL_W, Inches(0.4), titulo, tam=16 if compacta else tam, negrita=True)
    if subtitulo:
        caja_texto(d, MARGEN, Inches(y_sub), UTIL_W, Inches(0.4), subtitulo, tam=12, color=TINTA_2)
    caja_texto(d, ANCHO - MARGEN - Inches(0.6), Inches(7.1), Inches(0.6), Inches(0.24), str(len(prs.slides)),
               tam=8, color=TINTA_3, alinear=PP_ALIGN.RIGHT)
    return d


def tabla_conc(t: pd.DataFrame, nombre: str) -> pd.DataFrame:
    cols = [(ind, str(a_)) for ind in COLS_CONC for a_ in ANIOS_CONC.get(ind, (A2,))]
    out = a_texto(t[cols])
    out.columns = pd.MultiIndex.from_tuples([(corto(i).replace("% acumulado ajuste general", "% acum."), a_)
                                             for i, a_ in cols])
    out.index = [str(i)[:40] for i in out.index]
    out.index.name = nombre
    return out

## C.6 Informe de avance por región: definiciones

Solo las definiciones de esta sección que usa este notebook; la sección completa está en `Informe_Ajustes_Region.ipynb`.

In [ ]:
def nombre_corte(prefijo: str, vista: str) -> str:
    return (f"{prefijo}_{ARCHIVO[vista]}_{ETQ_ANT.replace(' ', '')}_vs_"
            f"{MES_ACT_TXT[:3].capitalize()}{INICIO_ACT.year}_al_{FECHA_MAX_D.day}.pptx")


def portada_corte(titulo: str, linea_2: str, soc):
    portada(titulo, f"{MES_ANT_TXT.capitalize()} {INICIO_ANT.year} (cerrado) vs {MES_ACT_TXT} al {FECHA_MAX_D.day} "
                    f"(en curso) · {nombre_sociedad(soc)}", f"{linea_2} · datos a {FECHA_MAX_D:%d/%m/%Y}",
            COLOR_SOC[soc])

## C.7 Res por canal de venta

Un PowerPoint por región y canal de venta (`CANALES_VENTA_TOP`: 8 en total) para los materiales de `FOCO`, mes
anterior cerrado vs mes en curso al corte. Después de la portada, la lámina de tendencia general (venta neta, precio
neto por kg y % cumplimiento del precio de lista general, día a día, de los materiales de `FOCO` juntos) y, por
material:

| Láminas | Contenido |
|---|---|
| General | Tabla del canal (Mayoreo + Puntos de Venta = Total del canal) y la gráfica de rangos del canal |
| Mayoreo / Puntos de Venta | La gráfica de rangos de cada canal de distribución con venta |
| Top distribuidoras | Las `TOP_DISTRIBUIDORAS` con más ajuste general, con su % acumulado |
| Top clientes | Los `TOP_CLIENTES` con más ajuste general, con su canal, distribuidora y el % de ajuste del mes anterior |

In [ ]:
def cumplimiento(texto: str) -> str:
    """En los PowerPoint de Res, '% aplicado al precio de lista' se lee '% cumplimiento del precio de lista' (en los
    encabezados cortos, '% aplicado' queda '% cumplimiento')."""
    return (texto.replace("% aplicado al precio de lista", "% cumplimiento del precio de lista")
            .replace("% aplicado", "% cumplimiento"))


def figura_tendencia_res(soc, cv, figsize=(15, 7.6)):
    """Por dia, los materiales de FOCO juntos en el canal de venta cv: venta neta, precio neto por kg y % cumplimiento
    del precio de lista general (mes anterior y mes en curso; la linea punteada separa los meses)."""
    f = foco_dia_d[(foco_dia_d.index.get_level_values("Sociedad") == soc)
                   & (foco_dia_d.index.get_level_values("Canal de Venta") == cv)]
    t = indicadores(f.groupby(level="Dia").sum()).reindex(DIAS_EJE)
    t.index = range(len(DIAS_EJE))
    paneles = [(VENTA_NETA, "Venta neta por día ($)", pesos, lambda v, _: "$" + num(v, 0)),
               (PRECIO_NETO, "Precio neto por kg, por día", pesos, lambda v, _: pesos(v)),
               (L_GEN, "% cumplimiento del precio de lista general por día (venta neta / venta bruta)", pct,
                lambda v, _: f"{v:,.0f}%")]
    fig, ejes = plt.subplots(len(paneles), 1, figsize=figsize, sharex=True)
    for ax, (col, titulo, formato, formato_y) in zip(ejes, paneles):
        s = t[col].dropna()   # los dias sin venta no cortan la linea
        ax.plot(s.index, s, color=COLOR_SOC[soc], linewidth=2, marker="o", markersize=3.5)
        for x in (s.index[[0, -1]] if len(s) else []):   # valor del primer y del ultimo dia
            ax.annotate(formato(s[x]), (x, s[x]), xytext=(0, 6), textcoords="offset points", ha="center",
                        va="bottom", fontsize=7, color="#0b0b0b")
        ejes_limpios(ax, formato_y=formato_y)
        ax.margins(y=0.2)
        ax.axvline(CORTE_ANIO, color="#8c8b87", linestyle=":", linewidth=1)
        ax.set_title(titulo, loc="left", fontsize=10)
    ejes[-1].set_xticks(range(len(DIAS_EJE)), ETIQ_MES, rotation=90)
    fig.suptitle(f"{' + '.join(FOCO)} · {cv} — {CORTO[soc]}: tendencia día a día ({ETQ_ANT} y {ETQ_ACT})",
                 x=0.01, ha="left", fontsize=12)
    plt.tight_layout()
    return fig


def lamina_res_tendencia(vista, soc, cv, nota):
    """Lamina 2: tendencia diaria de los materiales de FOCO juntos en el canal de venta (foco_d ya filtrado con
    solo_canal)."""
    f = foco_d[foco_d.index.get_level_values("Sociedad") == soc]
    ind = indicadores(f.groupby(level="Año").sum()).reindex(ANIOS)
    d = lamina(f"Res · {cv}: precio neto por kg {pesos(ind.loc[A1, PRECIO_NETO])} → {pesos(ind.loc[A2, PRECIO_NETO])}"
               f" · % cumplimiento del precio de lista general {pct(ind.loc[A1, L_GEN])} → {pct(ind.loc[A2, L_GEN])}",
               f"{vista} · {cv} · {' + '.join(FOCO)} · venta neta {pesos(ind.loc[A1, VENTA_NETA])} → "
               f"{pesos(ind.loc[A2, VENTA_NETA])} · {nota}", f"Tendencia general · {' + '.join(FOCO)}")
    imagen(d, figura_tendencia_res(soc, cv))


def lamina_foco_general(vista, soc, m, cv, seccion, nota):
    """Tabla del material en el canal de venta: Mayoreo + Puntos de Venta = Total del canal."""
    t = foco_canal_cd(soc, m)
    tot = t.loc["Total"]
    abiertas = [i for i in t.index if i != "Total"]
    kg = t[(VENTA_KG, A2)]
    mayor = kg[abiertas].idxmax() if kg[abiertas].notna().any() else None
    d = lamina(f"{m} · {cv}: % cumplimiento del precio de lista general {pct(tot[(L_GEN, A1)])} → {pct(tot[(L_GEN, A2)])}",
               f"{vista} · " + (f"{mayor} es el {pct(kg[mayor] / kg['Total'] * 100)} de la venta kg de {A2} · "
                                if mayor else "") + f"Mayoreo + Puntos de Venta = Total del canal · {nota}", seccion)
    txt = a_texto(t)
    txt.columns = pd.MultiIndex.from_tuples([(cumplimiento(corto(i)), a_) for i, a_ in t.columns])
    tabla(d, txt, y=Inches(1.7), tam=10, ancho_indice=0.14, alto_fila=Inches(0.4), calor=calor_ajustes(t, txt.index),
          negrita_filas=["Total"])


def lamina_foco_rangos(vista, soc, m, seccion, canal_dist=None, canal_venta=None):
    """Grafica de rangos (como la seccion 3 del informe general) del material en el canal de venta, y si se da, en un
    canal de distribucion."""
    c = foco_de(soc, m)
    nombre = m
    if canal_venta:
        c = c[c.index.get_level_values("Canal de Venta") == canal_venta]
        nombre += f" · {canal_venta}"
    if canal_dist:
        c = c[c.index.get_level_values("Canal de Distribucion") == canal_dist]
        nombre += f" · {ROTULO_CANAL.get(canal_dist, canal_dist)}"
    ind = indicadores(c.groupby(level="Año").sum()).reindex(ANIOS)
    bajada = (f"{vista} · % cumplimiento del precio de lista general {pct(ind.loc[A1, L_GEN])} → "
              f"{pct(ind.loc[A2, L_GEN])} · ajuste de precio de mercado {pct(ind.loc[A1, L_AJ])} → "
              f"{pct(ind.loc[A2, L_AJ])} · fecha corta {pct(ind.loc[A1, L_FC])} → {pct(ind.loc[A2, L_FC])} · venta kg "
              f"{num(ind.loc[A1, VENTA_KG], 0)} → {num(ind.loc[A2, VENTA_KG], 0)}")
    d = lamina(f"{nombre}: {frase_rango(c)}", bajada, seccion)
    imagen(d, figura_rangos(c, f"% de la venta bruta por rango de % cumplimiento del precio de lista general y motivo "
                               f"— {nombre} (al lado de cada barra: % y ajuste general, y el % de cada motivo)",
                            figsize=(14, 7.4), detalle=True))


def lamina_foco_distribuidoras(vista, soc, m, seccion, filas=TOP_DISTRIBUIDORAS):
    t = foco_distribuidoras(soc, m)
    a2 = str(A2)   # el pareto rotula los periodos como texto
    if t.empty or not t[(P_GEN, a2)].gt(0).any():
        lamina(f"{m}: sin ajuste general en {A2}", vista, seccion)
        return
    mitad = list(t.index[: int((t[("% acumulado ajuste general", a2)] < 50).sum()) + 1])
    top = t.head(filas)
    d = lamina(f"{m}: {len(mitad)} distribuidoras concentran la mitad del ajuste general de {A2} — "
               + ", ".join(n_.title() for n_ in mitad[:6]),
               f"{vista} · top {len(top)} de {len(t)} distribuidoras con venta, ordenadas por ajuste general de {A2} · "
               f"ajuste general: {pesos(t[(P_GEN, a2)].sum())}", seccion)
    txt = tabla_conc(top, "Distribuidora")
    txt.columns = pd.MultiIndex.from_tuples([(cumplimiento(i), a_) for i, a_ in txt.columns])
    tabla(d, txt, tam=9 if len(top) <= 14 else 7.5,
          negrita_filas=[str(i)[:40] for i in mitad], alto_fila=Inches(min(0.33, 5.4 / (len(top) + 2))),
          anchos=ANCHOS_CONC)


def lamina_foco_clientes(vista, soc, m, seccion):
    tc, part, total, n_cli = foco_clientes(soc, m)
    if tc is None:
        lamina(f"{m}: sin clientes con ajuste en {A2}", vista, seccion)
        return
    txt = a_texto(tc)
    txt.columns = [corto(c_) for c_ in tc.columns]
    txt.index = [str(i)[:34] for i in txt.index]
    txt.index.name = "Cliente"
    sujeto = ("el único cliente con ajuste concentra" if len(tc) == 1
              else f"los {len(tc)} clientes con más ajuste general suman")
    d = lamina(f"{m}: {sujeto} el {pct(part)} del ajuste general de {A2}",
               f"{vista} · ajuste general: {pesos(total)} · {num(n_cli, 0)} clientes con ajuste · "
               f"ordenados por ajuste general en pesos · al lado, el % de ajuste general de {A1}", seccion)
    tabla(d, txt, y=Inches(1.5), tam=7, ancho_indice=0.19, alto_fila=Inches(0.17), calor=calor_ajustes(tc, txt.index))


def laminas_foco_canal(vista, soc, cv, nota):
    """Por material, en el canal de venta cv (foco_d ya filtrado con solo_canal): tabla general del canal, grafica de
    rangos del canal, una grafica por canal de distribucion, top de distribuidoras y top de clientes."""
    texto = f"{vista} · {cv}"
    for n, m in enumerate(FOCO, start=1):
        sec_ = f"{n}. {m} · {cv}"
        if foco_de(soc, m)["Venta Bruta"].sum() <= 0:
            lamina(f"{m}: sin venta en {cv}", texto, sec_)
            continue
        lamina_foco_general(texto, soc, m, cv, f"{sec_} · General", nota)
        lamina_foco_rangos(texto, soc, m, f"{sec_} · General", canal_venta=cv)
        for cd in canales_dist_foco(soc, m):
            lamina_foco_rangos(texto, soc, m, f"{sec_} · {ROTULO_CANAL.get(cd, cd)}", canal_dist=cd, canal_venta=cv)
        lamina_foco_distribuidoras(texto, soc, m, f"{sec_} · Top distribuidoras")
        lamina_foco_clientes(texto, soc, m, f"{sec_} · Top clientes")


def informe_foco_res(vista: str, cv: str):
    """Version mes anterior cerrado vs mes en curso, un canal de venta."""
    soc = VISTAS[vista]
    with modo_corte(), solo_canal(cv):
        nueva_presentacion()
        portada_corte(f"Res · {cv}: {' y '.join(FOCO)}",
                      "Tendencia diaria, general del canal, Mayoreo, Puntos de Venta y top de distribuidoras y "
                      "clientes", soc)
        dias = dias_con_venta(soc)
        nota = f"kg y pesos de {A2} parciales ({dias[A2]} días con venta frente a {dias[A1]})"
        lamina_res_tendencia(vista, soc, cv, nota)
        laminas_foco_canal(vista, soc, cv, nota)
        guardar(nombre_corte("Res", vista).replace(ARCHIVO[vista], f"{ARCHIVO[vista]}_{cv.replace(' ', '_')}"))


for vista in VISTAS:
    for cv in CANALES_VENTA_TOP:
        informe_foco_res(vista, cv)